In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
df = pd.read_csv("data/netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
netflix = df.copy()

netflix['director'] = netflix['director'].fillna('Unknown')
netflix['cast'] = netflix['cast'].fillna('Unknown')
netflix['country'] = netflix['country'].fillna('Unknown')
netflix['rating'] = netflix['rating'].fillna('Unknown')
netflix['date_added'] = netflix['date_added'].fillna('Unknown')

In [4]:
movies = netflix[['title', 'director', 'cast', 'listed_in', 'description']].copy()

movies['tags'] = (
    movies['director'] + " " +
    movies['cast'] + " " +
    movies['listed_in'] + " " +
    movies['description']
)

movies = movies[['title', 'tags']]

movies['tags'] = movies['tags'].str.lower()

movies.head()

,title,tags
0,Dick Johnson Is Dead,kirsten johnson unknown documentaries as her f...
1,Blood & Water,"unknown ama qamata, khosi ngema, gail mabalane..."
2,Ganglands,"julien leclercq sami bouajila, tracy gotoas, s..."
3,Jailbirds New Orleans,"unknown unknown docuseries, reality tv feuds, ..."
4,Kota Factory,"unknown mayur more, jitendra kumar, ranjan raj..."


In [5]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(movies['tags']).toarray()

vectors.shape

(8807, 5000)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
similarity.shape

(8807, 8807)

In [9]:
movies[movies['title'] == "Stranger Things"].index


RangeIndex(start=3685, stop=3686, step=1)

In [11]:
def recommend(movie):

    if movie not in movies['title'].values:
        print("Movie not found!")
        return

    movie_index = movies[movies['title'] == movie].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print(f"\nRecommendations for '{movie}':\n")

    for i in movie_list:
        print(movies.iloc[i[0]].title)
recommend("Stranger Things")
        


Recommendations for 'Stranger Things':

Beyond Stranger Things
Warrior Nun
Nightflyers
Anjaan: Special Crimes Unit
The Umbrella Academy


In [12]:
import pickle

pickle.dump(movies, open("movie_list.pkl", "wb"))
pickle.dump(similarity, open("similarity.pkl", "wb"))